# NB11 — DistilBERT reference baseline (untuned)

**Why this notebook exists.** Reviewers in 2026 will ask how the COA-tuned FastText models compare with a
pre-trained transformer. This notebook fine-tunes `distilbert-base-uncased` once per dataset with standard,
*untuned* settings, as a reference point. It is not part of the COA study and is not hyperparameter-searched.

**Same data, same splits.** The transformer needs raw text (NB01 only stores stopword-removed train text), so the
raw data are reloaded with NB01's exact loader code and NB01's saved `train_idx` / `val_idx` are re-applied.
Labels are checked against NB01's saved `y_train` / `y_val` / `y_test`; the notebook stops if they disagree.
Training uses train+validation with the last 10% as the hold-out monitor, exactly like every other final model.

**Inputs (Add Input):**
- Datasets: `kritanjalijain/amazon-reviews`, `irustandi/yelp-review-polarity`, `kazanova/sentiment140` (same as NB01)
- Notebook output: `01-preprocessing` (split indices + labels), optionally `04-final-training` (for the summary table)
- Model: Internet ON (downloads `distilbert-base-uncased`), **or** add the Kaggle model
  `distilbert-base-uncased` as an input — the notebook finds it automatically.

**Outputs:** `saved_results/{dataset}/distilbert_metrics.json`, `distilbert_prob.npy`, `distilbert_train_log.json`,
`saved_results/global/distilbert_summary.csv`.

Accelerator: GPU T4 x2. Expected run time: about 2–3 hours for all three datasets. A per-dataset training-time
guard (120 + 150 + 60 min = 5.5 h at most, plus evaluation) keeps the notebook well inside one 9-hour Kaggle
session; if a guard ever triggers, training stops early and the log records how many samples were seen.

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
DATASETS        = ["amazon", "yelp", "twitter"]   # run a subset if you need to split across sessions
MODEL_NAME      = "distilbert-base-uncased"
MAX_LEN         = {"amazon": 128, "yelp": 256, "twitter": 64}   # subword tokens
EPOCHS          = 2
LR              = 2e-5
WEIGHT_DECAY    = 0.01
WARMUP_FRAC     = 0.06
BATCH_PER_GPU   = 32
EVAL_BATCH      = 256
EVALS_PER_EPOCH = 2          # hold-out checks per epoch; best checkpoint by hold-out loss is kept
TIME_BUDGET_MIN = {"amazon": 120, "yelp": 150, "twitter": 60}    # training-time guard per dataset (5.5 h in total)
SEED            = 42

KAGGLE_USER = "abhaysharma01702"
def nb_input(slug):
    return f"/kaggle/input/notebooks/{KAGGLE_USER}/{slug}"

In [ ]:
import os, sys, json, time, random, gc, glob, subprocess
os.environ["TOKENIZERS_PARALLELISM"] = "false"
import numpy as np
import pandas as pd

try:
    import transformers
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers"], check=True)
    import transformers

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from sklearn.metrics import (f1_score, accuracy_score, precision_score, recall_score,
    matthews_corrcoef, roc_auc_score, average_precision_score, classification_report)

def set_seeds(seed=SEED):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
set_seeds()

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
N_GPUS = torch.cuda.device_count()
print(f"torch {torch.__version__}  transformers {transformers.__version__}  GPUs={N_GPUS}")

WORKING     = "/kaggle/working"
RESULTS_DIR = f"{WORKING}/saved_results"
for ds in DATASETS + ["global"]:
    os.makedirs(f"{RESULTS_DIR}/{ds}", exist_ok=True)

NB01_SHARED  = f"{nb_input('01-preprocessing')}/shared_data"
NB01_RESULTS = f"{nb_input('01-preprocessing')}/saved_results"
NB04_RESULTS = f"{nb_input('04-final-training')}/saved_results"

# Model location: local Kaggle model input if present, otherwise the Hub (needs Internet ON)
def find_local_model():
    for cfg in glob.glob("/kaggle/input/**/config.json", recursive=True):
        d = os.path.dirname(cfg)
        has_tokenizer = (os.path.exists(os.path.join(d, "tokenizer_config.json"))
                         or os.path.exists(os.path.join(d, "vocab.txt")))
        if "distilbert" in d.lower() and has_tokenizer:
            return d
    return None
MODEL_SRC = find_local_model() or MODEL_NAME
print(f"Model source: {MODEL_SRC}")

In [ ]:
# ── Raw-data loaders copied from NB01 (must stay identical so NB01's indices line up) ──
AMAZON_TRAIN = "/kaggle/input/datasets/kritanjalijain/amazon-reviews/train.csv"
AMAZON_TEST  = "/kaggle/input/datasets/kritanjalijain/amazon-reviews/test.csv"
YELP_DIR     = "/kaggle/input/datasets/irustandi/yelp-review-polarity/yelp_review_polarity_csv"
TWITTER_PATH = "/kaggle/input/datasets/kazanova/sentiment140/training.1600000.processed.noemoticon.csv"

def load_amazon():
    tr = pd.read_csv(AMAZON_TRAIN, header=None, names=["label","title","text"])
    te = pd.read_csv(AMAZON_TEST,  header=None, names=["label","title","text"])
    tr["label"] = tr["label"].astype(int) - 1
    te["label"] = te["label"].astype(int) - 1
    pos = tr[tr["label"]==1].sample(n=300000, random_state=SEED)
    neg = tr[tr["label"]==0].sample(n=300000, random_state=SEED)
    tr  = pd.concat([pos,neg]).sample(frac=1, random_state=SEED).reset_index(drop=True)
    te  = te.sample(frac=1, random_state=SEED).reset_index(drop=True)
    tr_t = (tr["title"].fillna("")+" "+tr["text"].fillna("")).tolist()
    te_t = (te["title"].fillna("")+" "+te["text"].fillna("")).tolist()
    return tr_t, tr["label"].values, te_t, te["label"].values

def load_yelp():
    tr = pd.read_csv(f"{YELP_DIR}/train.csv", header=None, names=["label","text"])
    te = pd.read_csv(f"{YELP_DIR}/test.csv",  header=None, names=["label","text"])
    tr["label"] = tr["label"].astype(int) - 1
    te["label"] = te["label"].astype(int) - 1
    tr = tr.sample(frac=1, random_state=SEED).reset_index(drop=True)
    te = te.sample(frac=1, random_state=SEED).reset_index(drop=True)
    return tr["text"].fillna("").tolist(), tr["label"].values, te["text"].fillna("").tolist(), te["label"].values

def load_twitter(n=300000):
    df = pd.read_csv(TWITTER_PATH, encoding="latin-1", header=None,
                     names=["label","id","date","query","user","text"])
    df["label"] = (df["label"]==4).astype(int)
    pos = df[df["label"]==1].sample(n=n, random_state=SEED)
    neg = df[df["label"]==0].sample(n=n, random_state=SEED)
    df  = pd.concat([pos,neg]).sample(frac=1, random_state=SEED).reset_index(drop=True)
    n_te = int(len(df)*0.2)
    te = df.iloc[:n_te].reset_index(drop=True)
    tr = df.iloc[n_te:].reset_index(drop=True)
    return tr["text"].fillna("").tolist(), tr["label"].values, te["text"].fillna("").tolist(), te["label"].values

LOADERS = {"amazon": load_amazon, "yelp": load_yelp, "twitter": lambda: load_twitter(300000)}

def load_splits(ds):
    """Raw-text train/val/test using NB01's saved indices; aborts if labels disagree with NB01."""
    tr_t, tr_y, te_t, te_y = LOADERS[ds]()
    d = f"{NB01_SHARED}/{ds}"
    tr_idx, val_idx = np.load(f"{d}/train_idx.npy"), np.load(f"{d}/val_idx.npy")
    y_train, y_val, y_test = np.load(f"{d}/y_train.npy"), np.load(f"{d}/y_val.npy"), np.load(f"{d}/y_test.npy")
    X_tr = [tr_t[i] for i in tr_idx]; X_vl = [tr_t[i] for i in val_idx]
    checks = {"train": np.array_equal(tr_y[tr_idx], y_train),
              "val"  : np.array_equal(tr_y[val_idx], y_val),
              "test" : np.array_equal(te_y, y_test)}
    print(f"  split/label check vs NB01: {checks}")
    if not all(checks.values()):
        raise RuntimeError(f"{ds}: reconstructed splits do not match NB01 — do not use these results.")
    return X_tr, y_train, X_vl, y_val, te_t, y_test

In [ ]:
# ── Evaluation helper (same metrics as the TF notebooks) ──────────────────────
def evaluate(name, y_true, y_pred, y_prob=None):
    auc = round(roc_auc_score(y_true, y_prob), 4) if y_prob is not None else float("nan")
    ap  = round(average_precision_score(y_true, y_prob), 4) if y_prob is not None else float("nan")
    rep = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
    return {
        "Model"        : name,
        "Accuracy"     : round(accuracy_score(y_true, y_pred), 4),
        "Precision"    : round(precision_score(y_true, y_pred, zero_division=0), 4),
        "Recall"       : round(recall_score(y_true, y_pred, zero_division=0), 4),
        "F1-score"     : round(f1_score(y_true, y_pred, zero_division=0), 4),
        "Macro-F1"     : round(rep["macro avg"]["f1-score"], 4),
        "Weighted-F1"  : round(rep["weighted avg"]["f1-score"], 4),
        "MCC"          : round(matthews_corrcoef(y_true, y_pred), 4),
        "AUC-ROC"      : auc,
        "Avg-Precision": ap,
    }

In [ ]:
# ── Torch plumbing ────────────────────────────────────────────────────────────
class TextDS(Dataset):
    def __init__(self, texts, labels): self.t, self.y = texts, labels
    def __len__(self): return len(self.t)
    def __getitem__(self, i): return self.t[i], int(self.y[i])

def make_collate(tok, max_len):
    def collate(batch):
        texts, ys = zip(*batch)
        enc = tok(list(texts), truncation=True, max_length=max_len, padding=True, return_tensors="pt")
        return enc["input_ids"], enc["attention_mask"], torch.tensor(ys, dtype=torch.long)
    return collate

class AmpLogits(nn.Module):
    """Runs the HF model under autocast inside forward (needed for DataParallel) and returns logits only."""
    def __init__(self, m): super().__init__(); self.m = m
    def forward(self, input_ids, attention_mask):
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=torch.cuda.is_available()):
            return self.m(input_ids=input_ids, attention_mask=attention_mask).logits

@torch.no_grad()
def predict(model, loader):
    model.eval(); probs, losses, n = [], 0.0, 0
    ce = nn.CrossEntropyLoss(reduction="sum")
    for ids, mask, y in loader:
        ids, mask, y = ids.to(DEVICE), mask.to(DEVICE), y.to(DEVICE)
        logits = model(ids, mask).float()
        losses += ce(logits, y).item(); n += len(y)
        probs.append(torch.softmax(logits, dim=-1)[:, 1].cpu().numpy())
    model.train()
    return np.concatenate(probs), losses / max(n, 1)

In [ ]:
# ── Fine-tune + evaluate, one dataset at a time ───────────────────────────────
summary = []
for ds in DATASETS:
    print(f"\n{'='*60}\nDISTILBERT | {ds.upper()}\n{'='*60}")
    set_seeds()
    X_tr, y_tr, X_vl, y_vl, X_te, y_te = load_splits(ds)
    X_tv = X_tr + X_vl; y_tv = np.concatenate([y_tr, y_vl])
    n_es = int(len(y_tv) * 0.1)                       # same hold-out rule as the TF notebooks
    X_fit, y_fit, X_hold, y_hold = X_tv[:-n_es], y_tv[:-n_es], X_tv[-n_es:], y_tv[-n_es:]
    print(f"  fit={len(y_fit):,}  hold-out={len(y_hold):,}  test={len(y_te):,}  max_len={MAX_LEN[ds]}")

    tok = AutoTokenizer.from_pretrained(MODEL_SRC)
    base = AutoModelForSequenceClassification.from_pretrained(MODEL_SRC, num_labels=2)
    model = AmpLogits(base).to(DEVICE)
    if N_GPUS > 1:
        model = nn.DataParallel(model)
    collate = make_collate(tok, MAX_LEN[ds])
    g = torch.Generator(); g.manual_seed(SEED)
    bs = BATCH_PER_GPU * max(N_GPUS, 1)
    # num_workers=0: tokenizing a batch takes ~1 ms, far less than a GPU step, and it avoids all
    # multiprocessing pitfalls (pickling closures, forked tokenizers) at no real speed cost.
    pin = torch.cuda.is_available()
    fit_dl  = DataLoader(TextDS(X_fit, y_fit), batch_size=bs, shuffle=True, generator=g,
                         collate_fn=collate, num_workers=0, pin_memory=pin)
    hold_dl = DataLoader(TextDS(X_hold, y_hold), batch_size=EVAL_BATCH, shuffle=False,
                         collate_fn=collate, num_workers=0, pin_memory=pin)
    test_dl = DataLoader(TextDS(X_te, y_te), batch_size=EVAL_BATCH, shuffle=False,
                         collate_fn=collate, num_workers=0, pin_memory=pin)

    no_decay = ["bias", "LayerNorm.weight"]
    params = [{"params": [p for n, p in base.named_parameters() if not any(nd in n for nd in no_decay)],
               "weight_decay": WEIGHT_DECAY},
              {"params": [p for n, p in base.named_parameters() if any(nd in n for nd in no_decay)],
               "weight_decay": 0.0}]
    opt = torch.optim.AdamW(params, lr=LR)
    total_steps = EPOCHS * len(fit_dl)
    sched = get_linear_schedule_with_warmup(opt, int(WARMUP_FRAC * total_steps), total_steps)
    try:
        scaler = torch.amp.GradScaler("cuda", enabled=torch.cuda.is_available())
    except (AttributeError, TypeError):          # older torch
        scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())
    ce = nn.CrossEntropyLoss()
    eval_every = max(1, len(fit_dl) // EVALS_PER_EPOCH)

    best_loss, best_state, log, step, seen = float("inf"), None, [], 0, 0
    t0 = time.time(); stopped_early = False
    for epoch in range(EPOCHS):
        for ids, mask, y in fit_dl:
            ids, mask, y = ids.to(DEVICE), mask.to(DEVICE), y.to(DEVICE)
            logits = model(ids, mask).float()
            loss = ce(logits, y)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(base.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sched.step()
            step += 1; seen += len(y)
            if step % 200 == 0:
                print(f"    step {step}/{total_steps}  loss={loss.item():.4f}  {(time.time()-t0)/60:.1f} min")
            over_time = (time.time() - t0) / 60 > TIME_BUDGET_MIN[ds]
            if step % eval_every == 0 or step == total_steps or over_time:
                hp, hl = predict(model, hold_dl)
                hf1 = f1_score(y_hold, (hp > 0.5).astype(int))
                log.append({"step": step, "epoch": round(step / len(fit_dl), 3),
                            "samples_seen": seen, "holdout_loss": round(hl, 5),
                            "holdout_f1": round(float(hf1), 4), "minutes": round((time.time()-t0)/60, 1)})
                print(f"    [hold-out] step {step}: loss={hl:.4f}  F1={hf1:.4f}")
                if hl < best_loss:
                    best_loss = hl
                    best_state = {k: v.detach().cpu().clone() for k, v in base.state_dict().items()}
            if over_time:
                stopped_early = True
                print(f"    Time budget of {TIME_BUDGET_MIN[ds]} min reached — stopping training.")
                break
        if stopped_early:
            break

    base.load_state_dict(best_state)
    probs, _ = predict(model, test_dl)
    res = evaluate("DistilBERT (untuned)", y_te, (probs > 0.5).astype(int), probs)
    train_info = {"dataset": ds, "model": MODEL_SRC, "max_len": MAX_LEN[ds], "epochs_planned": EPOCHS,
                  "steps_done": step, "steps_planned": total_steps, "samples_seen": seen,
                  "stopped_by_time_budget": stopped_early, "best_holdout_loss": round(best_loss, 5),
                  "train_minutes": round((time.time()-t0)/60, 1), "lr": LR, "batch": bs,
                  "warmup_frac": WARMUP_FRAC, "weight_decay": WEIGHT_DECAY, "log": log}
    r_dir = f"{RESULTS_DIR}/{ds}"
    np.save(f"{r_dir}/distilbert_prob.npy", probs)
    json.dump(res, open(f"{r_dir}/distilbert_metrics.json", "w"), indent=2)
    json.dump(train_info, open(f"{r_dir}/distilbert_train_log.json", "w"), indent=2)
    print(f"  TEST: F1={res['F1-score']}  MCC={res['MCC']}  AUC={res['AUC-ROC']}  "
          f"(steps {step}/{total_steps}, early-stop-by-time={stopped_early})")
    summary.append({"Dataset": ds.capitalize(), **{k: res[k] for k in ["F1-score","Macro-F1","MCC","AUC-ROC"]},
                    "steps_done": step, "steps_planned": total_steps, "time_stopped": stopped_early})

    del model, base, opt, sched, best_state, fit_dl, hold_dl, test_dl, tok
    del X_tr, X_vl, X_te, X_tv, X_fit, X_hold
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

In [ ]:
# ── Summary next to the existing models (if NB01/NB04 outputs are linked) ─────
rows = []
for s in summary:
    ds = s["Dataset"].lower()
    rows.append({"Dataset": s["Dataset"], "Model": "DistilBERT (untuned)",
                 **{k: s[k] for k in ["F1-score","Macro-F1","MCC","AUC-ROC"]}})
    try:
        base = json.load(open(f"{NB01_RESULTS}/{ds}/baseline_metrics.json"))
        for name in ["TF-IDF + SVM", "TF-IDF + LR"]:
            rows.append({"Dataset": s["Dataset"], "Model": name,
                         **{k: base[name][k] for k in ["F1-score","Macro-F1","MCC","AUC-ROC"]}})
        for f, name in [("cnn_coa_metrics.json", "COA-CNN+BiLSTM"), ("ann_coa_metrics.json", "COA-ANN")]:
            m = json.load(open(f"{NB04_RESULTS}/{ds}/{f}"))
            rows.append({"Dataset": s["Dataset"], "Model": name,
                         **{k: m[k] for k in ["F1-score","Macro-F1","MCC","AUC-ROC"]}})
    except FileNotFoundError as e:
        print(f"  (summary: {e.filename} not linked — showing DistilBERT only for {ds})")
sum_df = pd.DataFrame(rows)
sum_df.to_csv(f"{RESULTS_DIR}/global/distilbert_summary.csv", index=False)
print(sum_df.to_string(index=False))